# 02 — Model Training
**Plant Disease Detection (Residual CBM)** — Step 2 of 4

Self-contained notebook: no `scripts/` import. Trains the two-headed Concept
Bottleneck Model — MobileNetV3Small (alpha=0.75) backbone, a 4-unit sigmoid
**concept** head, and a 15-unit softmax **disease** head that consumes the
concept head's output (the residual/concat connection).

Reads `manifest.csv` from Notebook 01 — does not recompute the split.

**Prerequisites:** `manifest.csv` present. GPU strongly recommended.


In [ ]:
import os
import ast

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Concatenate, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
import albumentations as A
import matplotlib.pyplot as plt

print(f"[INFO] TensorFlow {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
print(f"[INFO] GPUs visible to TensorFlow: {len(gpus)}", gpus if gpus else "(CPU only — training will be slow)")


In [ ]:
# --- Dataset constants (must match Notebook 01 exactly -- label indices and
# concept vectors are baked into manifest.csv and must not drift) ---
IMG_SIZE = 224
SEED = 42
SELECTED_CLASSES = [
    "Pepper__bell___Bacterial_spot", "Pepper__bell___healthy", "Potato___Early_blight",
    "Potato___Late_blight", "Potato___healthy", "Tomato_Bacterial_spot",
    "Tomato_Early_blight", "Tomato_Late_blight", "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot", "Tomato_Spider_mites_Two_spotted_spider_mite",
    "Tomato__Target_Spot", "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato__Tomato_mosaic_virus", "Tomato_healthy",
]
N_CONCEPTS = 4
N_CLASSES = len(SELECTED_CLASSES)
print(f"[INFO] {N_CLASSES} disease classes, {N_CONCEPTS} concept dimensions")


## 2.1 Load the manifest built in Notebook 01

In [ ]:
manifest_csv_path = "manifest.csv" if os.path.exists("manifest.csv") else "../manifest.csv"
manifest = pd.read_csv(manifest_csv_path)
manifest["concept_vec"] = manifest["concept_vec"].apply(ast.literal_eval)

train_df = manifest[manifest["split"] == "train"]
val_df = manifest[manifest["split"] == "val"]

print(f"[INFO] Train rows: {len(train_df)} | Val rows: {len(val_df)}")
print(f"[INFO] Concept vector length: {len(train_df['concept_vec'].iloc[0])}")
print(f"[INFO] Disease classes: {train_df['label_idx'].nunique()}")


## 2.2 tf.data pipeline (self-contained decode/resize + albumentations)

In [ ]:
def _decode_resize(path, img_size=IMG_SIZE):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, [img_size, img_size])
    return img

def get_tf_dataset(df, split_name, batch_size=32, shuffle=False, augment_fn=None, seed=SEED):
    paths = df["File_Path"].values
    concepts = np.stack(df["concept_vec"].values).astype(np.float32)
    labels = df["label_idx"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, concepts, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=seed, reshuffle_each_iteration=True)

    def _load(path, concept, label):
        img = _decode_resize(path)
        if augment_fn is not None:
            img = augment_fn(img)
        return img, (concept, label)

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# "Spatial sabotage": aggressive augmentation intended to force the network to
# rely on lesion texture/shape rather than memorizing background/lighting.
transform_train = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=45, p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.CoarseDropout(num_holes_range=(1, 5), hole_height_range=(10, 30), hole_width_range=(10, 30), p=0.5),
])

def train_aug_fn(img_tensor):
    def _aug(img):
        return transform_train(image=img.astype(np.uint8))["image"].astype(np.float32)
    out = tf.numpy_function(_aug, [img_tensor], tf.float32)
    out.set_shape(img_tensor.shape)
    return out

BATCH_SIZE = 32
train_ds = get_tf_dataset(train_df, "train", batch_size=BATCH_SIZE, shuffle=True, augment_fn=train_aug_fn)
val_ds = get_tf_dataset(val_df, "val", batch_size=BATCH_SIZE, shuffle=False, augment_fn=None)

imgs, (concepts, labels) = next(iter(train_ds))
print(f"[SANITY CHECK] image batch shape: {imgs.shape}, dtype: {imgs.dtype}")
print(f"[SANITY CHECK] concept batch shape: {concepts.shape}  (sigmoid targets in [0,1]: {bool(tf.reduce_all((concepts >= 0) & (concepts <= 1)))})")
print(f"[SANITY CHECK] label batch shape: {labels.shape}, unique labels in batch: {sorted(set(labels.numpy().tolist()))}")


## 2.3 Model architecture — Residual Concept Bottleneck Model

In [ ]:
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet", alpha=0.75
)
base_model.trainable = True

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.40)(x)

c_out = Dense(N_CONCEPTS, activation="sigmoid", name="C", kernel_regularizer=l2(0.01))(x)

combined = Concatenate()([x, c_out])
d_out = Dense(N_CLASSES, activation="softmax", name="D", kernel_regularizer=l2(0.01))(combined)

model = Model(inputs=base_model.input, outputs=[c_out, d_out], name="plant_disease_cbm")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=["binary_crossentropy", "sparse_categorical_crossentropy"],
    loss_weights=[0.4, 1.0],
    metrics=[["accuracy"], ["accuracy"]],
)

print(f"[INFO] Total params: {model.count_params():,}")
model.summary()


## 2.4 Train with EarlyStopping + ModelCheckpoint

In [ ]:
model_out_path = "best_model.keras" if os.path.exists("manifest.csv") else "../best_model.keras"
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_D_accuracy", mode="max", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(model_out_path, monitor="val_D_accuracy", mode="max", save_best_only=True),
]

print("[OPERATION] Starting training (early stopping decides the final epoch count)...")
history = model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks)
print(f"\n[SUCCESS] Best checkpoint saved to {model_out_path}")


## 2.5 Training curves

In [ ]:
hist_df = pd.DataFrame(history.history)
print("[RESULT] Final epoch metrics:")
print(hist_df.tail(1).T)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
hist_df[["D_accuracy", "val_D_accuracy"]].plot(ax=axes[0], marker="o")
axes[0].set_title("Disease-head accuracy")
axes[0].set_xlabel("epoch")

hist_df[["loss", "val_loss"]].plot(ax=axes[1], marker="o")
axes[1].set_title("Total loss (concept + disease, weighted)")
axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.show()

best_epoch = int(hist_df["val_D_accuracy"].idxmax())
print(f"[RESULT] Best val_D_accuracy = {hist_df['val_D_accuracy'].max():.4f} at epoch {best_epoch + 1}")


## 2.6 Summary
**Next notebook:** `03_Evaluation_Robustness_and_Quantization.ipynb`.
